In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install Ultralytics
!pip install ultralytics -q

# 3. Copy Dataset and Unzip silently
!cp "/content/drive/MyDrive/RoadEye_Workspace_V6/RoadEye_V6_Dataset.zip" "/content/"
!unzip -q /content/RoadEye_V6_Dataset.zip -d /content/dataset_v6

# 4. Generate YAML (Make sure this matches exactly what we used)
import yaml
yaml_content = {
    "path": "/content/dataset_v6",
    "train": "RoadEye_V6_Dataset/train/images",
    "val": "RoadEye_V6_Dataset/valid/images",
    "test": "RoadEye_V6_Dataset/test/images",
    "names": {0: "Crack", 1: "Pothole"}
}
yaml_path = "/content/roadeye_v6.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print("✅ Workspace ready for resuming training.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.5 MB/s eta 0:00:00
✅ Workspace ready for resuming training.


In [ ]:
# ==============================================================================
# RESUME ROADEYE V6.1B - CONTROLLED FINE-TUNING
# ==============================================================================
from ultralytics import YOLO

# 1. Point to the last saved checkpoint of V6.1B
# Ensure this path matches exactly where V6.1B was saving on your Drive
checkpoint_path = "/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6.1B_Controlled_FineTuning/weights/last.pt"

# 2. Load the model from the checkpoint
model = YOLO(checkpoint_path)

# 3. Resume training
# YOLOv8 will automatically read the args.yaml in the save_dir and continue
# with the exact same hyperparameters (epochs=50, lr, augmentations, etc.)
results = model.train(resume=True)

print("🏆 Training successfully resumed!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
WARNING ⚠️ model '/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6.1B_Controlled_FineTuning/weights/last.pt' is not a resumable training checkpoint (missing epoch/optimizer state). Use 'resume' only to continue incomplete training. Starting new training instead.
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco8.yaml, degrees=0.0, deterministic=True,

KeyboardInterrupt: 

In [ ]:
import os
import cv2
import math
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import shutil

# ==========================================================
# CONFIGURATION & PATHS
# ==========================================================
# Pointing to the latest V6.1B Fine-Tuned Model
MODEL_PATH = "/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6.1B_Controlled_FineTuning/weights/best.pt"
DATA_YAML_PATH = "/content/roadeye_v6.yaml"

TEST_IMAGES_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/images")
TEST_LABELS_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/labels")

GALLERY_DIR = Path("/content/RoadEye_Error_Gallery_V6_1B")
FP_DIR = GALLERY_DIR / "False_Positives"
FN_DIR = GALLERY_DIR / "False_Negatives"

IOU_THRESHOLD = 0.5
CONF_THRESHOLD = 0.25
MAX_GALLERY_IMAGES = 20

# Reset Gallery Directories for V6.1B
if GALLERY_DIR.exists():
    shutil.rmtree(GALLERY_DIR)
FP_DIR.mkdir(parents=True, exist_ok=True)
FN_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================================
# METRICS TRACKER
# ==========================================================
stats = {
    "class": {0: {"TP": 0, "FP": 0, "FN": 0}, 1: {"TP": 0, "FP": 0, "FN": 0}},
    "country": {"Japan": {"TP": 0, "FP": 0, "FN": 0}, "India": {"TP": 0, "FP": 0, "FN": 0}},
    "country_class": {
        "Japan": {0: {"TP":0, "FP":0, "FN":0}, 1: {"TP":0, "FP":0, "FN":0}},
        "India": {0: {"TP":0, "FP":0, "FN":0}, 1: {"TP":0, "FP":0, "FN":0}}
    },
    "size": {"16-32": {"TP": 0, "FN": 0}, "32-64": {"TP": 0, "FN": 0},
             "64-128": {"TP": 0, "FN": 0}, "128+": {"TP": 0, "FN": 0}},
    "density": {"1": {"TP": 0, "FN": 0}, "2": {"TP": 0, "FN": 0},
                "3-5": {"TP": 0, "FN": 0}, "6+": {"TP": 0, "FN": 0}},
    "aspect": {"tall": {"TP": 0, "FN": 0}, "square": {"TP": 0, "FN": 0}, "wide": {"TP": 0, "FN": 0}},
    "conf_dist": {"0.25-0.4": 0, "0.4-0.6": 0, "0.6-0.8": 0, "0.8-1.0": 0}
}

fp_gallery = []
fn_gallery = []

def box_iou(box1, box2):
    """Calculate IoU between two bounding boxes [cx, cy, w, h] (normalized)."""
    x1_1, y1_1 = box1[0] - box1[2]/2, box1[1] - box1[3]/2
    x2_1, y2_1 = box1[0] + box1[2]/2, box1[1] + box1[3]/2
    x1_2, y1_2 = box2[0] - box2[2]/2, box2[1] - box2[3]/2
    x2_2, y2_2 = box2[0] + box2[2]/2, box2[1] + box2[3]/2
    xi1, yi1 = max(x1_1, x1_2), max(y1_1, y1_2)
    xi2, yi2 = min(x2_1, x2_2), min(y2_1, y2_2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union_area = (box1[2]*box1[3]) + (box2[2]*box2[3]) - inter_area
    return inter_area / union_area if union_area > 0 else 0

def get_geom_attrs(w_norm, h_norm, img_w, img_h):
    """Extract geometric bins for detailed analysis."""
    size = math.sqrt((w_norm * img_w) * (h_norm * img_h))
    if size < 32: s_bin = "16-32"
    elif size < 64: s_bin = "32-64"
    elif size < 128: s_bin = "64-128"
    else: s_bin = "128+"

    ratio = w_norm / h_norm if h_norm > 0 else 1
    if ratio > 1.5: a_bin = "wide"
    elif ratio < 0.67: a_bin = "tall"
    else: a_bin = "square"
    return s_bin, a_bin, size

def calc_metrics(tp, fp, fn):
    """Calculate Precision, Recall, and F1-Score."""
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return p * 100, r * 100, f1 * 100

def draw_and_save_error(img_path, box_norm, cls_id, save_dir, prefix, metric_val, is_fp=True):
    """Draw bounding boxes on error instances and save them to the gallery."""
    img = cv2.imread(str(img_path))
    if img is None: return
    h, w = img.shape[:2]

    cx, cy, bw, bh = box_norm
    x1 = int((cx - bw/2) * w)
    y1 = int((cy - bh/2) * h)
    x2 = int((cx + bw/2) * w)
    y2 = int((cy + bh/2) * h)

    color = (255, 0, 0) if is_fp else (0, 0, 255)
    label = f"FP {cls_id} (Conf: {metric_val:.2f})" if is_fp else f"FN {cls_id} (Size: {metric_val:.0f}px)"

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
    cv2.putText(img, label, (x1, max(10, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    save_name = f"{prefix}_{img_path.name}"
    cv2.imwrite(str(save_dir / save_name), img)

def run_forensics():
    print("🚀 [PHASE 1] Loading Model & Running Official mAP Evaluation on Test Set...")
    model = YOLO(MODEL_PATH)

    # Official validation strictly on the test set
    val_metrics = model.val(
        data=DATA_YAML_PATH,
        split="test",
        imgsz=1024,
        batch=4,
        device=0,
        verbose=False,
        plots=False
    )

    official_map50 = val_metrics.box.map50 * 100
    official_map50_95 = val_metrics.box.map * 100
    official_crack_map50 = val_metrics.box.maps[0] * 100
    official_pothole_map50 = val_metrics.box.maps[1] * 100

    print("🚀 [PHASE 2] Starting Custom Forensic Analysis...")
    image_paths = list(TEST_IMAGES_DIR.glob("*.*"))

    for idx, img_path in enumerate(image_paths):
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']: continue

        lbl_path = TEST_LABELS_DIR / f"{img_path.stem}.txt"
        gt_boxes = {0: [], 1: []}

        img = cv2.imread(str(img_path))
        if img is None: continue
        img_h, img_w = img.shape[:2]

        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if len(parts) >= 5: gt_boxes[int(parts[0])].append(parts[1:5])

        total_gt = len(gt_boxes[0]) + len(gt_boxes[1])
        density_bin = "1" if total_gt == 1 else "2" if total_gt == 2 else "3-5" if 3 <= total_gt <= 5 else "6+"
        country = "Japan" if img_path.name.startswith("jp_") else "India"

        results = model.predict(source=str(img_path), imgsz=1024, conf=CONF_THRESHOLD, verbose=False)
        pred_data = results[0].boxes
        pred_boxes = {0: [], 1: []}

        if pred_data is not None and len(pred_data) > 0:
            boxes_norm = pred_data.xywhn.cpu().numpy()
            classes = pred_data.cls.cpu().numpy()
            confs = pred_data.conf.cpu().numpy()
            for i in range(len(classes)):
                pred_boxes[int(classes[i])].append((boxes_norm[i], confs[i]))
                c_val = confs[i]
                if 0.25 <= c_val < 0.4: stats["conf_dist"]["0.25-0.4"] += 1
                elif 0.4 <= c_val < 0.6: stats["conf_dist"]["0.4-0.6"] += 1
                elif 0.6 <= c_val < 0.8: stats["conf_dist"]["0.6-0.8"] += 1
                else: stats["conf_dist"]["0.8-1.0"] += 1

        for cls_id in [0, 1]:
            gts = gt_boxes[cls_id]
            preds = sorted(pred_boxes[cls_id], key=lambda x: x[1], reverse=True)
            matched_gt = set()

            for p_box, conf in preds:
                best_iou = 0
                best_gt_idx = -1
                for gt_idx, g_box in enumerate(gts):
                    if gt_idx in matched_gt: continue
                    iou = box_iou(p_box, g_box)
                    if iou > best_iou:
                        best_iou, best_gt_idx = iou, gt_idx

                if best_iou >= IOU_THRESHOLD:
                    matched_gt.add(best_gt_idx)
                    stats["class"][cls_id]["TP"] += 1
                    stats["country"][country]["TP"] += 1
                    stats["country_class"][country][cls_id]["TP"] += 1

                    s_bin, a_bin, _ = get_geom_attrs(gts[best_gt_idx][2], gts[best_gt_idx][3], img_w, img_h)
                    stats["size"][s_bin]["TP"] += 1
                    stats["aspect"][a_bin]["TP"] += 1
                    if total_gt > 0: stats["density"][density_bin]["TP"] += 1
                else:
                    stats["class"][cls_id]["FP"] += 1
                    stats["country"][country]["FP"] += 1
                    stats["country_class"][country][cls_id]["FP"] += 1
                    fp_gallery.append((conf, img_path, p_box, cls_id))

            for gt_idx, g_box in enumerate(gts):
                if gt_idx not in matched_gt:
                    stats["class"][cls_id]["FN"] += 1
                    stats["country"][country]["FN"] += 1
                    stats["country_class"][country][cls_id]["FN"] += 1

                    s_bin, a_bin, raw_size = get_geom_attrs(g_box[2], g_box[3], img_w, img_h)
                    stats["size"][s_bin]["FN"] += 1
                    stats["aspect"][a_bin]["FN"] += 1
                    if total_gt > 0: stats["density"][density_bin]["FN"] += 1
                    fn_gallery.append((raw_size, img_path, g_box, cls_id))

        if idx % 100 == 0: print(f"   -> Analyzed {idx}/{len(image_paths)} images...")

    print("\n📸 Generating Error Gallery for V6.1B...")
    fp_gallery.sort(key=lambda x: x[0], reverse=True)
    fn_gallery.sort(key=lambda x: x[0], reverse=True)

    for i, (conf, p, b, c) in enumerate(fp_gallery[:MAX_GALLERY_IMAGES]):
        draw_and_save_error(p, b, c, FP_DIR, f"top{i+1}_fp", conf, is_fp=True)
    for i, (sz, p, b, c) in enumerate(fn_gallery[:MAX_GALLERY_IMAGES]):
        draw_and_save_error(p, b, c, FN_DIR, f"top{i+1}_fn", sz, is_fp=False)

    print_report(official_map50, official_map50_95, official_crack_map50, official_pothole_map50)

def print_report(map50, map50_95, crack_map50, pothole_map50):
    print("\n" + "="*70)
    print(" 🚨 ROADEYE V6.1B ULTIMATE FORENSIC REPORT 🚨")
    print("="*70)

    print("\n[ 0. Official Test Set mAP Scores (Ultralytics Engine) ]")
    print(f"   Overall mAP50    : {map50:.2f}%")
    print(f"   Overall mAP50-95 : {map50_95:.2f}%")
    print(f"   Crack mAP50      : {crack_map50:.2f}%")
    print(f"   Pothole mAP50    : {pothole_map50:.2f}%")
    print("-" * 70)

    print("\n[ 1. Class Analysis (Threshold = 0.25) ]")
    print(f"{'Category':<10} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
    print("-" * 70)
    for name, cid in [("Crack", 0), ("Pothole", 1)]:
        s = stats["class"][cid]
        p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
        print(f"{name:<10} | {s['TP']:<5} | {s['FP']:<5} | {s['FN']:<5} | {p:>6.1f}% | {r:>6.1f}% | {f1:>6.1f}%")

    print("\n[ 2. Country Analysis ]")
    print(f"{'Country':<10} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
    print("-" * 70)
    for c in ["Japan", "India"]:
        s = stats["country"][c]
        p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
        print(f"{c:<10} | {s['TP']:<5} | {s['FP']:<5} | {s['FN']:<5} | {p:>6.1f}% | {r:>6.1f}% | {f1:>6.1f}%")

    print("\n[ 3. Per-Country Per-Class Analysis (⭐⭐⭐⭐⭐) ]")
    print(f"{'Country':<8} | {'Class':<8} | {'TP':<4} | {'FP':<4} | {'FN':<4} | {'Prec.':<6} | {'Recall':<6} | {'F1':<6}")
    print("-" * 70)
    for c in ["Japan", "India"]:
        for name, cid in [("Crack", 0), ("Pothole", 1)]:
            s = stats["country_class"][c][cid]
            p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
            print(f"{c:<8} | {name:<8} | {s['TP']:<4} | {s['FP']:<4} | {s['FN']:<4} | {p:>5.1f}% | {r:>5.1f}% | {f1:>5.1f}%")

    print("\n[ 4. Confidence Distribution (Model Certainty) ]")
    for b, c in stats["conf_dist"].items(): print(f"   Conf {b:<8}: {c} predictions")

    print("\n[ 5. Geometric Recalls (Size & Aspect Ratio) ]")
    print("   [Size]")
    for s in ["16-32", "32-64", "64-128", "128+"]:
        r = calc_metrics(stats["size"][s]["TP"], 0, stats["size"][s]["FN"])[1]
        print(f"      {s:<8}: {r:.1f}%")
    print("   [Aspect Ratio]")
    for a in ["tall", "square", "wide"]:
        r = calc_metrics(stats["aspect"][a]["TP"], 0, stats["aspect"][a]["FN"])[1]
        print(f"      {a:<8}: {r:.1f}%")

    print("\n[ 6. Density Recall ]")
    for d in ["1", "2", "3-5", "6+"]:
        r = calc_metrics(stats["density"][d]["TP"], 0, stats["density"][d]["FN"])[1]
        print(f"   {d:<8} objects : {r:.1f}%")

    print("\n📸 Error Gallery Exported! Check '/content/RoadEye_Error_Gallery_V6_1B'")
    print("="*70)

if __name__ == "__main__":
    run_forensics()

🚀 [PHASE 1] Loading Model & Running Official mAP Evaluation on Test Set...
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,918 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 12.7±5.8 MB/s, size: 75.7 KB)
val: Scanning /content/dataset_v6/RoadEye_V6_Dataset/test/labels... 746 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 746/746 454.9it/s 1.6s
val: New cache created: /content/dataset_v6/RoadEye_V6_Dataset/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 187/187 3.1it/s 1:01
                   all        746       1608      0.618       0.61      0.614      0.279
Speed: 2.7ms preprocess, 74.5ms inference, 0.0ms loss, 1.3ms postprocess per image
🚀 [PHASE 2] Starting Custom Forensic Analysis...
   -> Analyzed 0/746 images...
   -> Analyzed 100/746 images...
   -> Analyzed 200/746 images...
   ->

In [ ]:
import shutil
from google.colab import files

# ==========================================================
# ZIP AND DOWNLOAD V6.1B ERROR GALLERY TO LOCAL MACHINE
# ==========================================================

gallery_dir = "/content/RoadEye_Error_Gallery_V6_1B"
zip_file_path = "/content/RoadEye_Error_Gallery_V6_1B.zip"

print("📦 Archiving V6.1B Error Gallery...")

# 1. Archive the entire V6.1B gallery directory into a zip file
shutil.make_archive(gallery_dir, 'zip', gallery_dir)

# 2. Trigger the browser to download the zip file
print("⬇️ Triggering download...")
files.download(zip_file_path)

print("✅ Download triggered! Check your browser's downloads.")

In [ ]:
import os
import cv2
import math
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import shutil

# ==========================================================
# CONFIGURATION & PATHS
# ==========================================================
# Pointing to the latest V6.1B Fine-Tuned Model
MODEL_PATH = "/content/drive/MyDrive/RoadEye_Workspace_V6/Runs/V6.1B_Controlled_FineTuning/weights/best.pt"
DATA_YAML_PATH = "/content/roadeye_v6.yaml"

TEST_IMAGES_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/images")
TEST_LABELS_DIR = Path("/content/dataset_v6/RoadEye_V6_Dataset/test/labels")

GALLERY_DIR = Path("/content/RoadEye_Error_Gallery_V6_1B_TTA")
FP_DIR = GALLERY_DIR / "False_Positives"
FN_DIR = GALLERY_DIR / "False_Negatives"

IOU_THRESHOLD = 0.5
CONF_THRESHOLD = 0.25
MAX_GALLERY_IMAGES = 20

# Reset Gallery Directories for V6.1B TTA
if GALLERY_DIR.exists():
    shutil.rmtree(GALLERY_DIR)
FP_DIR.mkdir(parents=True, exist_ok=True)
FN_DIR.mkdir(parents=True, exist_ok=True)

# ==========================================================
# METRICS TRACKER
# ==========================================================
stats = {
    "class": {0: {"TP": 0, "FP": 0, "FN": 0}, 1: {"TP": 0, "FP": 0, "FN": 0}},
    "country": {"Japan": {"TP": 0, "FP": 0, "FN": 0}, "India": {"TP": 0, "FP": 0, "FN": 0}},
    "country_class": {
        "Japan": {0: {"TP":0, "FP":0, "FN":0}, 1: {"TP":0, "FP":0, "FN":0}},
        "India": {0: {"TP":0, "FP":0, "FN":0}, 1: {"TP":0, "FP":0, "FN":0}}
    },
    "size": {"16-32": {"TP": 0, "FN": 0}, "32-64": {"TP": 0, "FN": 0},
             "64-128": {"TP": 0, "FN": 0}, "128+": {"TP": 0, "FN": 0}},
    "density": {"1": {"TP": 0, "FN": 0}, "2": {"TP": 0, "FN": 0},
                "3-5": {"TP": 0, "FN": 0}, "6+": {"TP": 0, "FN": 0}},
    "aspect": {"tall": {"TP": 0, "FN": 0}, "square": {"TP": 0, "FN": 0}, "wide": {"TP": 0, "FN": 0}},
    "conf_dist": {"0.25-0.4": 0, "0.4-0.6": 0, "0.6-0.8": 0, "0.8-1.0": 0}
}

fp_gallery = []
fn_gallery = []

def box_iou(box1, box2):
    """Calculate IoU between two bounding boxes [cx, cy, w, h] (normalized)."""
    x1_1, y1_1 = box1[0] - box1[2]/2, box1[1] - box1[3]/2
    x2_1, y2_1 = box1[0] + box1[2]/2, box1[1] + box1[3]/2
    x1_2, y1_2 = box2[0] - box2[2]/2, box2[1] - box2[3]/2
    x2_2, y2_2 = box2[0] + box2[2]/2, box2[1] + box2[3]/2
    xi1, yi1 = max(x1_1, x1_2), max(y1_1, y1_2)
    xi2, yi2 = min(x2_1, x2_2), min(y2_1, y2_2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union_area = (box1[2]*box1[3]) + (box2[2]*box2[3]) - inter_area
    return inter_area / union_area if union_area > 0 else 0

def get_geom_attrs(w_norm, h_norm, img_w, img_h):
    """Extract geometric bins for detailed analysis."""
    size = math.sqrt((w_norm * img_w) * (h_norm * img_h))
    if size < 32: s_bin = "16-32"
    elif size < 64: s_bin = "32-64"
    elif size < 128: s_bin = "64-128"
    else: s_bin = "128+"

    ratio = w_norm / h_norm if h_norm > 0 else 1
    if ratio > 1.5: a_bin = "wide"
    elif ratio < 0.67: a_bin = "tall"
    else: a_bin = "square"
    return s_bin, a_bin, size

def calc_metrics(tp, fp, fn):
    """Calculate Precision, Recall, and F1-Score."""
    p = tp / (tp + fp) if (tp + fp) > 0 else 0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return p * 100, r * 100, f1 * 100

def draw_and_save_error(img_path, box_norm, cls_id, save_dir, prefix, metric_val, is_fp=True):
    """Draw bounding boxes on error instances and save them to the gallery."""
    img = cv2.imread(str(img_path))
    if img is None: return
    h, w = img.shape[:2]

    cx, cy, bw, bh = box_norm
    x1 = int((cx - bw/2) * w)
    y1 = int((cy - bh/2) * h)
    x2 = int((cx + bw/2) * w)
    y2 = int((cy + bh/2) * h)

    color = (255, 0, 0) if is_fp else (0, 0, 255)
    label = f"FP {cls_id} (Conf: {metric_val:.2f})" if is_fp else f"FN {cls_id} (Size: {metric_val:.0f}px)"

    cv2.rectangle(img, (x1, y1), (x2, y2), color, 3)
    cv2.putText(img, label, (x1, max(10, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    save_name = f"{prefix}_{img_path.name}"
    cv2.imwrite(str(save_dir / save_name), img)

def run_forensics():
    print("🚀 [PHASE 1] Official mAP Evaluation on Test Set (TTA ENABLED)...")
    model = YOLO(MODEL_PATH)

    # Official validation strictly on the test set WITH TTA
    val_metrics = model.val(
        data=DATA_YAML_PATH,
        split="test",
        imgsz=1024,
        batch=4,
        device=0,
        augment=True,       # 🔴 TEST-TIME AUGMENTATION (TTA) ENABLED
        verbose=False,
        plots=False
    )

    official_map50 = val_metrics.box.map50 * 100
    official_map50_95 = val_metrics.box.map * 100
    official_crack_map50 = val_metrics.box.maps[0] * 100
    official_pothole_map50 = val_metrics.box.maps[1] * 100

    print("🚀 [PHASE 2] Custom Forensic Analysis (TTA ENABLED)...")
    image_paths = list(TEST_IMAGES_DIR.glob("*.*"))

    for idx, img_path in enumerate(image_paths):
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png']: continue

        lbl_path = TEST_LABELS_DIR / f"{img_path.stem}.txt"
        gt_boxes = {0: [], 1: []}

        img = cv2.imread(str(img_path))
        if img is None: continue
        img_h, img_w = img.shape[:2]

        if lbl_path.exists():
            with open(lbl_path, "r") as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if len(parts) >= 5: gt_boxes[int(parts[0])].append(parts[1:5])

        total_gt = len(gt_boxes[0]) + len(gt_boxes[1])
        density_bin = "1" if total_gt == 1 else "2" if total_gt == 2 else "3-5" if 3 <= total_gt <= 5 else "6+"
        country = "Japan" if img_path.name.startswith("jp_") else "India"

        # Predict WITH TTA
        results = model.predict(
            source=str(img_path),
            imgsz=1024,
            conf=CONF_THRESHOLD,
            augment=True,       # 🔴 TEST-TIME AUGMENTATION (TTA) ENABLED
            verbose=False
        )

        pred_data = results[0].boxes
        pred_boxes = {0: [], 1: []}

        if pred_data is not None and len(pred_data) > 0:
            boxes_norm = pred_data.xywhn.cpu().numpy()
            classes = pred_data.cls.cpu().numpy()
            confs = pred_data.conf.cpu().numpy()
            for i in range(len(classes)):
                pred_boxes[int(classes[i])].append((boxes_norm[i], confs[i]))
                c_val = confs[i]
                if 0.25 <= c_val < 0.4: stats["conf_dist"]["0.25-0.4"] += 1
                elif 0.4 <= c_val < 0.6: stats["conf_dist"]["0.4-0.6"] += 1
                elif 0.6 <= c_val < 0.8: stats["conf_dist"]["0.6-0.8"] += 1
                else: stats["conf_dist"]["0.8-1.0"] += 1

        for cls_id in [0, 1]:
            gts = gt_boxes[cls_id]
            preds = sorted(pred_boxes[cls_id], key=lambda x: x[1], reverse=True)
            matched_gt = set()

            for p_box, conf in preds:
                best_iou = 0
                best_gt_idx = -1
                for gt_idx, g_box in enumerate(gts):
                    if gt_idx in matched_gt: continue
                    iou = box_iou(p_box, g_box)
                    if iou > best_iou:
                        best_iou, best_gt_idx = iou, gt_idx

                if best_iou >= IOU_THRESHOLD:
                    matched_gt.add(best_gt_idx)
                    stats["class"][cls_id]["TP"] += 1
                    stats["country"][country]["TP"] += 1
                    stats["country_class"][country][cls_id]["TP"] += 1

                    s_bin, a_bin, _ = get_geom_attrs(gts[best_gt_idx][2], gts[best_gt_idx][3], img_w, img_h)
                    stats["size"][s_bin]["TP"] += 1
                    stats["aspect"][a_bin]["TP"] += 1
                    if total_gt > 0: stats["density"][density_bin]["TP"] += 1
                else:
                    stats["class"][cls_id]["FP"] += 1
                    stats["country"][country]["FP"] += 1
                    stats["country_class"][country][cls_id]["FP"] += 1
                    fp_gallery.append((conf, img_path, p_box, cls_id))

            for gt_idx, g_box in enumerate(gts):
                if gt_idx not in matched_gt:
                    stats["class"][cls_id]["FN"] += 1
                    stats["country"][country]["FN"] += 1
                    stats["country_class"][country][cls_id]["FN"] += 1

                    s_bin, a_bin, raw_size = get_geom_attrs(g_box[2], g_box[3], img_w, img_h)
                    stats["size"][s_bin]["FN"] += 1
                    stats["aspect"][a_bin]["FN"] += 1
                    if total_gt > 0: stats["density"][density_bin]["FN"] += 1
                    fn_gallery.append((raw_size, img_path, g_box, cls_id))

        if idx % 100 == 0: print(f"   -> Analyzed {idx}/{len(image_paths)} images...")

    print("\n📸 Generating TTA Error Gallery...")
    fp_gallery.sort(key=lambda x: x[0], reverse=True)
    fn_gallery.sort(key=lambda x: x[0], reverse=True)

    for i, (conf, p, b, c) in enumerate(fp_gallery[:MAX_GALLERY_IMAGES]):
        draw_and_save_error(p, b, c, FP_DIR, f"top{i+1}_fp_tta", conf, is_fp=True)
    for i, (sz, p, b, c) in enumerate(fn_gallery[:MAX_GALLERY_IMAGES]):
        draw_and_save_error(p, b, c, FN_DIR, f"top{i+1}_fn_tta", sz, is_fp=False)

    print_report(official_map50, official_map50_95, official_crack_map50, official_pothole_map50)

def print_report(map50, map50_95, crack_map50, pothole_map50):
    print("\n" + "="*70)
    print(" 🚨 ROADEYE V6.1B ULTIMATE FORENSIC REPORT (WITH TTA) 🚨")
    print("="*70)

    print("\n[ 0. Official Test Set mAP Scores (TTA ENABLED) ]")
    print(f"   Overall mAP50    : {map50:.2f}%")
    print(f"   Overall mAP50-95 : {map50_95:.2f}%")
    print(f"   Crack mAP50      : {crack_map50:.2f}%")
    print(f"   Pothole mAP50    : {pothole_map50:.2f}%")
    print("-" * 70)

    print("\n[ 1. Class Analysis (Threshold = 0.25) ]")
    print(f"{'Category':<10} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
    print("-" * 70)
    for name, cid in [("Crack", 0), ("Pothole", 1)]:
        s = stats["class"][cid]
        p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
        print(f"{name:<10} | {s['TP']:<5} | {s['FP']:<5} | {s['FN']:<5} | {p:>6.1f}% | {r:>6.1f}% | {f1:>6.1f}%")

    print("\n[ 2. Country Analysis ]")
    print(f"{'Country':<10} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Prec.':<7} | {'Recall':<7} | {'F1-Score':<7}")
    print("-" * 70)
    for c in ["Japan", "India"]:
        s = stats["country"][c]
        p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
        print(f"{c:<10} | {s['TP']:<5} | {s['FP']:<5} | {s['FN']:<5} | {p:>6.1f}% | {r:>6.1f}% | {f1:>6.1f}%")

    print("\n[ 3. Per-Country Per-Class Analysis (⭐⭐⭐⭐⭐) ]")
    print(f"{'Country':<8} | {'Class':<8} | {'TP':<4} | {'FP':<4} | {'FN':<4} | {'Prec.':<6} | {'Recall':<6} | {'F1':<6}")
    print("-" * 70)
    for c in ["Japan", "India"]:
        for name, cid in [("Crack", 0), ("Pothole", 1)]:
            s = stats["country_class"][c][cid]
            p, r, f1 = calc_metrics(s["TP"], s["FP"], s["FN"])
            print(f"{c:<8} | {name:<8} | {s['TP']:<4} | {s['FP']:<4} | {s['FN']:<4} | {p:>5.1f}% | {r:>5.1f}% | {f1:>5.1f}%")

    print("\n[ 4. Confidence Distribution (Model Certainty) ]")
    for b, c in stats["conf_dist"].items(): print(f"   Conf {b:<8}: {c} predictions")

    print("\n[ 5. Geometric Recalls (Size & Aspect Ratio) ]")
    print("   [Size]")
    for s in ["16-32", "32-64", "64-128", "128+"]:
        r = calc_metrics(stats["size"][s]["TP"], 0, stats["size"][s]["FN"])[1]
        print(f"      {s:<8}: {r:.1f}%")
    print("   [Aspect Ratio]")
    for a in ["tall", "square", "wide"]:
        r = calc_metrics(stats["aspect"][a]["TP"], 0, stats["aspect"][a]["FN"])[1]
        print(f"      {a:<8}: {r:.1f}%")

    print("\n[ 6. Density Recall ]")
    for d in ["1", "2", "3-5", "6+"]:
        r = calc_metrics(stats["density"][d]["TP"], 0, stats["density"][d]["FN"])[1]
        print(f"   {d:<8} objects : {r:.1f}%")

    print("\n📸 Error Gallery Exported! Check '/content/RoadEye_Error_Gallery_V6_1B_TTA'")
    print("="*70)

if __name__ == "__main__":
    run_forensics()